# arc1withgen-aug-1000 statistics

`data/arc1withgen-aug-1000` の train / test split それぞれについて、puzzle、group、sequence length、grid shape、identifier、token 分布を集計して可視化します。

train split は `.npy` が大きいため、per-example の巨大な DataFrame は作らず、mmap と chunk 集計で処理します。token 分布は既定では局所windowサンプリングです。正確な token histogram が必要な場合は `EXACT_TOKEN_HIST = True` に変更してください。

In [ ]:
from pathlib import Path
import json
import math
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

DATASET_DIR = Path("data/arc1withgen-aug-1000")
ORIGIN_INPUT_FILE_PREFIX = Path("kaggle/combined/arc-agi")

# ここを変更すると train / test を個別にも両方にも実行できます。
SPLITS = ["train", "test"]  # e.g. ["train"], ["test"], or ["train", "test"]

# train split を元データ subset ごとに分けて集計します。
ENABLE_TRAIN_ORIGIN_BREAKDOWN = True
TRAIN_ORIGIN_SUBSETS = ["training", "evaluation", "concept"]

# train は 9,482万 examples あるため、メモリに合わせて調整してください。
CHUNK_SIZE = 2_000_000

# token 分布は既定でサンプリング。正確に全 token を読む場合は EXACT_TOKEN_HIST=True。
TOKEN_SAMPLE_SIZE = 3_000_000
TOKEN_SAMPLE_WINDOW_SIZE = 250_000
EXACT_TOKEN_HIST = False
RANDOM_SEED = 42

TOP_N = 30
DEFAULT_MAX_GRID_SIZE = 30
ARC_VOCAB_SIZE = 12
ARC_TOKEN_NAMES = {
    0: "pad",
    1: "eos",
    2: "color_0_black",
    3: "color_1_blue",
    4: "color_2_red",
    5: "color_3_green",
    6: "color_4_yellow",
    7: "color_5_gray",
    8: "color_6_magenta",
    9: "color_7_orange",
    10: "color_8_cyan",
    11: "color_9_dark_red",
}
HEX_TASK_RE = re.compile(r"^[0-9a-f]{8}$")

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

## Load and aggregate helpers

In [ ]:
ARRAY_NAMES = [
    "inputs",
    "labels",
    "seq_offsets",
    "label_seq_offsets",
    "seq_shapes",
    "label_seq_shapes",
    "puzzle_indices",
    "group_indices",
    "puzzle_identifiers",
]


def load_identifiers(dataset_dir=DATASET_DIR):
    with (dataset_dir / "identifiers.json").open() as f:
        return json.load(f)


def load_split_arrays(split, dataset_dir=DATASET_DIR):
    split_dir = dataset_dir / split
    if not split_dir.exists():
        raise FileNotFoundError(f"split directory not found: {split_dir}")

    with (split_dir / "dataset.json").open() as f:
        metadata = json.load(f)

    arrays = {}
    for name in ARRAY_NAMES:
        path = split_dir / f"all__{name}.npy"
        if path.exists():
            arrays[name] = np.load(path, mmap_mode="r", allow_pickle=False)
    return {"split": split, "metadata": metadata, "arrays": arrays}


def iter_ranges(n, chunk_size):
    for start in range(0, int(n), int(chunk_size)):
        yield start, min(int(n), start + int(chunk_size))


def add_bincount(counts, values):
    values = np.asarray(values, dtype=np.int64)
    if values.size == 0:
        return counts
    local = np.bincount(values)
    if local.size > counts.size:
        counts = np.pad(counts, (0, local.size - counts.size))
    counts[: local.size] += local.astype(counts.dtype, copy=False)
    return counts


def count_hist_from_values(values):
    return add_bincount(np.zeros(1, dtype=np.int64), values)


def offset_length_hist(offsets, chunk_size=CHUNK_SIZE):
    n = len(offsets) - 1
    counts = np.zeros(1, dtype=np.int64)
    for start, end in iter_ranges(n, chunk_size):
        lengths = np.asarray(offsets[start + 1 : end + 1]) - np.asarray(offsets[start:end])
        counts = add_bincount(counts, lengths)
    return counts


def ensure_square_matrix(matrix, size):
    if matrix.shape[0] >= size:
        return matrix
    out = np.zeros((size, size), dtype=matrix.dtype)
    out[: matrix.shape[0], : matrix.shape[1]] = matrix
    return out


def shape_histograms(input_shapes, output_shapes, max_grid_size=DEFAULT_MAX_GRID_SIZE, chunk_size=CHUNK_SIZE):
    dim_bins = int(max_grid_size) + 1
    stats = {
        "input_h": np.zeros(dim_bins, dtype=np.int64),
        "input_w": np.zeros(dim_bins, dtype=np.int64),
        "output_h": np.zeros(dim_bins, dtype=np.int64),
        "output_w": np.zeros(dim_bins, dtype=np.int64),
        "input_area": np.zeros(int(max_grid_size) * int(max_grid_size) + 1, dtype=np.int64),
        "output_area": np.zeros(int(max_grid_size) * int(max_grid_size) + 1, dtype=np.int64),
        "area_delta": np.zeros(2 * int(max_grid_size) * int(max_grid_size) + 1, dtype=np.int64),
        "area_delta_offset": int(max_grid_size) * int(max_grid_size),
        "input_dim_heatmap": np.zeros((dim_bins, dim_bins), dtype=np.int64),
        "output_dim_heatmap": np.zeros((dim_bins, dim_bins), dtype=np.int64),
        "same_shape_count": 0,
        "same_area_count": 0,
        "num_examples": int(input_shapes.shape[0]),
    }

    for start, end in iter_ranges(input_shapes.shape[0], chunk_size):
        in_chunk = np.asarray(input_shapes[start:end], dtype=np.int64)
        out_chunk = np.asarray(output_shapes[start:end], dtype=np.int64)

        in_h, in_w = in_chunk[:, 0], in_chunk[:, 1]
        out_h, out_w = out_chunk[:, 0], out_chunk[:, 1]
        max_seen = int(max(in_h.max(initial=0), in_w.max(initial=0), out_h.max(initial=0), out_w.max(initial=0)))
        if max_seen >= dim_bins:
            dim_bins = max_seen + 1
            stats["input_h"] = np.pad(stats["input_h"], (0, dim_bins - stats["input_h"].size))
            stats["input_w"] = np.pad(stats["input_w"], (0, dim_bins - stats["input_w"].size))
            stats["output_h"] = np.pad(stats["output_h"], (0, dim_bins - stats["output_h"].size))
            stats["output_w"] = np.pad(stats["output_w"], (0, dim_bins - stats["output_w"].size))
            stats["input_dim_heatmap"] = ensure_square_matrix(stats["input_dim_heatmap"], dim_bins)
            stats["output_dim_heatmap"] = ensure_square_matrix(stats["output_dim_heatmap"], dim_bins)

        stats["input_h"] = add_bincount(stats["input_h"], in_h)
        stats["input_w"] = add_bincount(stats["input_w"], in_w)
        stats["output_h"] = add_bincount(stats["output_h"], out_h)
        stats["output_w"] = add_bincount(stats["output_w"], out_w)

        in_area = in_h * in_w
        out_area = out_h * out_w
        stats["input_area"] = add_bincount(stats["input_area"], in_area)
        stats["output_area"] = add_bincount(stats["output_area"], out_area)

        area_delta_indices = out_area - in_area + stats["area_delta_offset"]
        stats["area_delta"] = add_bincount(stats["area_delta"], area_delta_indices)

        in_linear = in_h * dim_bins + in_w
        out_linear = out_h * dim_bins + out_w
        stats["input_dim_heatmap"] += np.bincount(in_linear, minlength=dim_bins * dim_bins).reshape(dim_bins, dim_bins)
        stats["output_dim_heatmap"] += np.bincount(out_linear, minlength=dim_bins * dim_bins).reshape(dim_bins, dim_bins)

        stats["same_shape_count"] += int(np.count_nonzero((in_h == out_h) & (in_w == out_w)))
        stats["same_area_count"] += int(np.count_nonzero(in_area == out_area))
    return stats


def offset_length_hist_for_example_ranges(offsets, example_ranges, chunk_size=CHUNK_SIZE):
    counts = np.zeros(1, dtype=np.int64)
    for range_start, range_end in example_ranges:
        for start in range(int(range_start), int(range_end), int(chunk_size)):
            end = min(int(range_end), start + int(chunk_size))
            lengths = np.asarray(offsets[start + 1 : end + 1]) - np.asarray(offsets[start:end])
            counts = add_bincount(counts, lengths)
    return counts


def shape_histograms_for_example_ranges(input_shapes, output_shapes, example_ranges, max_grid_size=DEFAULT_MAX_GRID_SIZE, chunk_size=CHUNK_SIZE):
    dim_bins = int(max_grid_size) + 1
    stats = {
        "input_h": np.zeros(dim_bins, dtype=np.int64),
        "input_w": np.zeros(dim_bins, dtype=np.int64),
        "output_h": np.zeros(dim_bins, dtype=np.int64),
        "output_w": np.zeros(dim_bins, dtype=np.int64),
        "input_area": np.zeros(int(max_grid_size) * int(max_grid_size) + 1, dtype=np.int64),
        "output_area": np.zeros(int(max_grid_size) * int(max_grid_size) + 1, dtype=np.int64),
        "area_delta": np.zeros(2 * int(max_grid_size) * int(max_grid_size) + 1, dtype=np.int64),
        "area_delta_offset": int(max_grid_size) * int(max_grid_size),
        "input_dim_heatmap": np.zeros((dim_bins, dim_bins), dtype=np.int64),
        "output_dim_heatmap": np.zeros((dim_bins, dim_bins), dtype=np.int64),
        "same_shape_count": 0,
        "same_area_count": 0,
        "num_examples": int(sum(int(end) - int(start) for start, end in example_ranges)),
    }

    for range_start, range_end in example_ranges:
        for start in range(int(range_start), int(range_end), int(chunk_size)):
            end = min(int(range_end), start + int(chunk_size))
            in_chunk = np.asarray(input_shapes[start:end], dtype=np.int64)
            out_chunk = np.asarray(output_shapes[start:end], dtype=np.int64)
            if in_chunk.size == 0:
                continue

            in_h, in_w = in_chunk[:, 0], in_chunk[:, 1]
            out_h, out_w = out_chunk[:, 0], out_chunk[:, 1]
            max_seen = int(max(in_h.max(initial=0), in_w.max(initial=0), out_h.max(initial=0), out_w.max(initial=0)))
            if max_seen >= dim_bins:
                dim_bins = max_seen + 1
                stats["input_h"] = np.pad(stats["input_h"], (0, dim_bins - stats["input_h"].size))
                stats["input_w"] = np.pad(stats["input_w"], (0, dim_bins - stats["input_w"].size))
                stats["output_h"] = np.pad(stats["output_h"], (0, dim_bins - stats["output_h"].size))
                stats["output_w"] = np.pad(stats["output_w"], (0, dim_bins - stats["output_w"].size))
                stats["input_dim_heatmap"] = ensure_square_matrix(stats["input_dim_heatmap"], dim_bins)
                stats["output_dim_heatmap"] = ensure_square_matrix(stats["output_dim_heatmap"], dim_bins)

            stats["input_h"] = add_bincount(stats["input_h"], in_h)
            stats["input_w"] = add_bincount(stats["input_w"], in_w)
            stats["output_h"] = add_bincount(stats["output_h"], out_h)
            stats["output_w"] = add_bincount(stats["output_w"], out_w)

            in_area = in_h * in_w
            out_area = out_h * out_w
            stats["input_area"] = add_bincount(stats["input_area"], in_area)
            stats["output_area"] = add_bincount(stats["output_area"], out_area)
            stats["area_delta"] = add_bincount(stats["area_delta"], out_area - in_area + stats["area_delta_offset"])

            in_linear = in_h * dim_bins + in_w
            out_linear = out_h * dim_bins + out_w
            stats["input_dim_heatmap"] += np.bincount(in_linear, minlength=dim_bins * dim_bins).reshape(dim_bins, dim_bins)
            stats["output_dim_heatmap"] += np.bincount(out_linear, minlength=dim_bins * dim_bins).reshape(dim_bins, dim_bins)

            stats["same_shape_count"] += int(np.count_nonzero((in_h == out_h) & (in_w == out_w)))
            stats["same_area_count"] += int(np.count_nonzero(in_area == out_area))
    return stats


def token_ranges_from_example_ranges(offsets, example_ranges):
    return [
        (int(offsets[int(start)]), int(offsets[int(end)]))
        for start, end in example_ranges
        if int(end) > int(start)
    ]


def token_histogram_for_token_ranges(tokens, token_ranges, vocab_size=ARC_VOCAB_SIZE, exact=False, sample_size=TOKEN_SAMPLE_SIZE, window_size=TOKEN_SAMPLE_WINDOW_SIZE, rng=None):
    counts = np.zeros(int(vocab_size), dtype=np.int64)
    token_ranges = [(int(start), int(end)) for start, end in token_ranges if int(end) > int(start)]
    total_tokens = int(sum(end - start for start, end in token_ranges))
    rng = np.random.default_rng(RANDOM_SEED) if rng is None else rng

    def add_block(block, counts):
        local = np.bincount(np.asarray(block, dtype=np.int64), minlength=counts.size)
        if local.size > counts.size:
            counts = np.pad(counts, (0, local.size - counts.size))
        counts[: local.size] += local.astype(counts.dtype, copy=False)
        return counts

    if total_tokens == 0:
        return counts, {"mode": "empty", "tokens_analyzed": 0, "total_tokens": 0}

    if exact or sample_size is None or int(sample_size) >= total_tokens:
        for range_start, range_end in token_ranges:
            for start in range(range_start, range_end, CHUNK_SIZE * 8):
                end = min(range_end, start + CHUNK_SIZE * 8)
                counts = add_block(tokens[start:end], counts)
        return counts, {"mode": "exact", "tokens_analyzed": total_tokens, "total_tokens": total_tokens}

    lengths = np.array([end - start for start, end in token_ranges], dtype=np.int64)
    max_range_len = int(lengths.max(initial=0))
    remaining = max(0, int(sample_size))
    analyzed = 0
    while remaining > 0 and max_range_len > 0:
        size = min(int(window_size), remaining, max_range_len)
        eligible = np.flatnonzero(lengths >= size)
        if eligible.size == 0:
            size = int(max_range_len)
            eligible = np.flatnonzero(lengths >= size)
        weights = lengths[eligible].astype(np.float64)
        weights /= weights.sum()
        range_idx = int(rng.choice(eligible, p=weights))
        range_start, range_end = token_ranges[range_idx]
        start = int(rng.integers(range_start, range_end - size + 1))
        counts = add_block(tokens[start : start + size], counts)
        analyzed += size
        remaining -= size
    return counts, {"mode": "sampled_category_windows", "tokens_analyzed": analyzed, "total_tokens": total_tokens}


def token_histogram(tokens, vocab_size=ARC_VOCAB_SIZE, exact=False, sample_size=TOKEN_SAMPLE_SIZE, window_size=TOKEN_SAMPLE_WINDOW_SIZE, rng=None):
    total_tokens = int(tokens.shape[0])
    counts = np.zeros(int(vocab_size), dtype=np.int64)
    rng = np.random.default_rng(RANDOM_SEED) if rng is None else rng

    def add_block(block, counts):
        local = np.bincount(np.asarray(block, dtype=np.int64), minlength=counts.size)
        if local.size > counts.size:
            counts = np.pad(counts, (0, local.size - counts.size))
        counts[: local.size] += local.astype(counts.dtype, copy=False)
        return counts

    if exact or sample_size is None or int(sample_size) >= total_tokens:
        for start, end in iter_ranges(total_tokens, CHUNK_SIZE * 8):
            counts = add_block(tokens[start:end], counts)
        return counts, {"mode": "exact", "tokens_analyzed": total_tokens, "total_tokens": total_tokens}

    remaining = max(0, int(sample_size))
    analyzed = 0
    while remaining > 0 and total_tokens > 0:
        size = min(int(window_size), remaining, total_tokens)
        start = int(rng.integers(0, total_tokens - size + 1))
        counts = add_block(tokens[start : start + size], counts)
        analyzed += size
        remaining -= size
    return counts, {"mode": "sampled_windows", "tokens_analyzed": analyzed, "total_tokens": total_tokens}


def parse_identifier(name):
    parts = str(name).split("|||")
    base_repr = parts[0]
    source = None
    base_task = base_repr
    if ":" in base_repr:
        maybe_source, maybe_task = base_repr.split(":", 1)
        if maybe_source.startswith(("ARC-AGI", "ARC-GEN")):
            source = maybe_source
            base_task = maybe_task
    transform = parts[1] if len(parts) >= 2 else "original"
    source_like = source or ("ARC-AGI-like" if HEX_TASK_RE.fullmatch(base_task) else "ARC-GEN-like")
    is_augmented = len(parts) >= 2
    return base_task, transform, source_like, is_augmented


def identifier_histograms(puzzle_identifier_ids, identifiers):
    base_tasks = Counter()
    transforms = Counter()
    source_like = Counter()
    augmented = Counter()

    for identifier_id in np.asarray(puzzle_identifier_ids):
        idx = int(identifier_id)
        name = identifiers[idx] if 0 <= idx < len(identifiers) else f"<unknown:{idx}>"
        base_task, transform, source, is_augmented = parse_identifier(name)
        base_tasks[base_task] += 1
        transforms[transform] += 1
        source_like[source] += 1
        augmented["augmented" if is_augmented else "original"] += 1

    return {
        "base_tasks": base_tasks,
        "transforms": transforms,
        "source_like": source_like,
        "augmented": augmented,
    }


def load_origin_task_ids(input_file_prefix=ORIGIN_INPUT_FILE_PREFIX, subsets=TRAIN_ORIGIN_SUBSETS):
    task_ids_by_subset = {}
    for subset in subsets:
        path = Path(f"{input_file_prefix}_{subset}-challenges.json")
        if not path.exists():
            print(f"origin subset file not found: {path}")
            task_ids_by_subset[subset] = set()
            continue
        with path.open() as f:
            task_ids_by_subset[subset] = set(json.load(f).keys())
    return task_ids_by_subset


def categorize_puzzle_identifiers(puzzle_identifier_ids, identifiers, task_ids_by_subset):
    labels = list(task_ids_by_subset.keys()) + ["unknown"]
    unknown_code = len(labels) - 1
    task_to_code = {}
    for code, label in enumerate(labels[:-1]):
        for task_id in task_ids_by_subset[label]:
            task_to_code[task_id] = code

    codes = np.empty(len(puzzle_identifier_ids), dtype=np.int16)
    id_code_cache = {}
    for i, identifier_id in enumerate(np.asarray(puzzle_identifier_ids)):
        identifier_id = int(identifier_id)
        code = id_code_cache.get(identifier_id)
        if code is None:
            name = identifiers[identifier_id] if 0 <= identifier_id < len(identifiers) else f"<unknown:{identifier_id}>"
            base_task, _transform, _source, _is_augmented = parse_identifier(name)
            code = task_to_code.get(base_task, unknown_code)
            id_code_cache[identifier_id] = code
        codes[i] = code
    return codes, labels


def group_codes_from_puzzle_codes(puzzle_codes, group_indices):
    group_starts = np.asarray(group_indices[:-1], dtype=np.int64)
    group_ends = np.asarray(group_indices[1:], dtype=np.int64)
    group_codes = np.empty(len(group_starts), dtype=np.int16)
    mixed_groups = 0
    for group_id, (start, end) in enumerate(zip(group_starts, group_ends)):
        if start >= end:
            group_codes[group_id] = -1
            continue
        code = int(puzzle_codes[int(start)])
        group_codes[group_id] = code
        if np.any(puzzle_codes[int(start) : int(end)] != code):
            mixed_groups += 1
    if mixed_groups:
        print(f"warning: {mixed_groups} groups contain multiple origin categories")
    return group_codes


def merge_ranges(ranges):
    ranges = sorted((int(start), int(end)) for start, end in ranges if int(end) > int(start))
    if not ranges:
        return []
    merged = [ranges[0]]
    for start, end in ranges[1:]:
        prev_start, prev_end = merged[-1]
        if start <= prev_end:
            merged[-1] = (prev_start, max(prev_end, end))
        else:
            merged.append((start, end))
    return merged


def compute_train_origin_breakdown(identifiers, input_file_prefix=ORIGIN_INPUT_FILE_PREFIX, subsets=TRAIN_ORIGIN_SUBSETS, chunk_size=CHUNK_SIZE):
    data = load_split_arrays("train")
    arrays = data["arrays"]
    metadata = data["metadata"]
    task_ids_by_subset = load_origin_task_ids(input_file_prefix, subsets)

    puzzle_codes, labels = categorize_puzzle_identifiers(arrays["puzzle_identifiers"], identifiers, task_ids_by_subset)
    group_codes = group_codes_from_puzzle_codes(puzzle_codes, arrays["group_indices"])

    puzzle_indices = np.asarray(arrays["puzzle_indices"], dtype=np.int64)
    group_indices = np.asarray(arrays["group_indices"], dtype=np.int64)
    group_starts = group_indices[:-1]
    group_ends = group_indices[1:]
    puzzle_counts = np.diff(puzzle_indices)

    max_grid_size = int(round(math.sqrt(metadata.get("seq_len", DEFAULT_MAX_GRID_SIZE * DEFAULT_MAX_GRID_SIZE))))
    max_grid_size = max(DEFAULT_MAX_GRID_SIZE, max_grid_size)

    stats_by_origin = {}
    for code, label in enumerate(labels):
        puzzle_mask = puzzle_codes == code
        group_mask = group_codes == code
        if not np.any(puzzle_mask) and not np.any(group_mask):
            continue

        cat_group_starts = group_starts[group_mask]
        cat_group_ends = group_ends[group_mask]
        group_counts = cat_group_ends - cat_group_starts
        group_example_counts = puzzle_indices[cat_group_ends] - puzzle_indices[cat_group_starts]
        cat_puzzle_counts = puzzle_counts[puzzle_mask]
        example_ranges = merge_ranges(
            (puzzle_indices[start], puzzle_indices[end])
            for start, end in zip(cat_group_starts, cat_group_ends)
        )

        print(f"[train/{label}] sequence length histograms")
        input_length_hist = offset_length_hist_for_example_ranges(arrays["seq_offsets"], example_ranges, chunk_size=chunk_size)
        label_length_hist = offset_length_hist_for_example_ranges(arrays["label_seq_offsets"], example_ranges, chunk_size=chunk_size)

        print(f"[train/{label}] grid shape histograms")
        shapes = shape_histograms_for_example_ranges(
            arrays["seq_shapes"],
            arrays["label_seq_shapes"],
            example_ranges,
            max_grid_size=max_grid_size,
            chunk_size=chunk_size,
        )

        print(f"[train/{label}] identifier histograms")
        identifiers_stats = identifier_histograms(arrays["puzzle_identifiers"][puzzle_mask], identifiers)

        rng = np.random.default_rng(RANDOM_SEED + sum(ord(ch) for ch in label))
        print(f"[train/{label}] token histograms ({'exact' if EXACT_TOKEN_HIST else 'sampled'})")
        input_token_ranges = token_ranges_from_example_ranges(arrays["seq_offsets"], example_ranges)
        label_token_ranges = token_ranges_from_example_ranges(arrays["label_seq_offsets"], example_ranges)
        input_token_hist, input_token_info = token_histogram_for_token_ranges(
            arrays["inputs"],
            input_token_ranges,
            vocab_size=metadata.get("vocab_size", ARC_VOCAB_SIZE),
            exact=EXACT_TOKEN_HIST,
            sample_size=TOKEN_SAMPLE_SIZE,
            window_size=TOKEN_SAMPLE_WINDOW_SIZE,
            rng=rng,
        )
        label_token_hist, label_token_info = token_histogram_for_token_ranges(
            arrays["labels"],
            label_token_ranges,
            vocab_size=metadata.get("vocab_size", ARC_VOCAB_SIZE),
            exact=EXACT_TOKEN_HIST,
            sample_size=TOKEN_SAMPLE_SIZE,
            window_size=TOKEN_SAMPLE_WINDOW_SIZE,
            rng=rng,
        )

        stats_by_origin[label] = {
            "split": f"train/{label}",
            "metadata": metadata,
            "num_groups": int(np.count_nonzero(group_mask)),
            "num_puzzles": int(np.count_nonzero(puzzle_mask)),
            "num_examples": int(cat_puzzle_counts.sum()),
            "input_tokens_total": int(input_token_info["total_tokens"]),
            "label_tokens_total": int(label_token_info["total_tokens"]),
            "puzzles_per_group_hist": count_hist_from_values(group_counts),
            "examples_per_group_hist": count_hist_from_values(group_example_counts),
            "examples_per_puzzle_hist": count_hist_from_values(cat_puzzle_counts),
            "input_length_hist": input_length_hist,
            "label_length_hist": label_length_hist,
            "shapes": shapes,
            "identifiers": identifiers_stats,
            "input_token_hist": input_token_hist,
            "label_token_hist": label_token_hist,
            "input_token_info": input_token_info,
            "label_token_info": label_token_info,
        }
    return stats_by_origin


def compute_split_stats(split, identifiers, chunk_size=CHUNK_SIZE):
    data = load_split_arrays(split)
    arrays = data["arrays"]
    metadata = data["metadata"]
    rng = np.random.default_rng(RANDOM_SEED + sum(ord(ch) for ch in split))

    puzzle_counts = np.diff(np.asarray(arrays["puzzle_indices"], dtype=np.int64))
    group_counts = np.diff(np.asarray(arrays["group_indices"], dtype=np.int64))
    group_starts = np.asarray(arrays["group_indices"][:-1], dtype=np.int64)
    group_ends = np.asarray(arrays["group_indices"][1:], dtype=np.int64)
    group_example_counts = np.asarray(arrays["puzzle_indices"][group_ends], dtype=np.int64) - np.asarray(arrays["puzzle_indices"][group_starts], dtype=np.int64)

    max_grid_size = int(round(math.sqrt(metadata.get("seq_len", DEFAULT_MAX_GRID_SIZE * DEFAULT_MAX_GRID_SIZE))))
    max_grid_size = max(DEFAULT_MAX_GRID_SIZE, max_grid_size)

    print(f"[{split}] sequence length histograms")
    input_length_hist = offset_length_hist(arrays["seq_offsets"], chunk_size=chunk_size)
    label_length_hist = offset_length_hist(arrays["label_seq_offsets"], chunk_size=chunk_size)

    print(f"[{split}] grid shape histograms")
    shapes = shape_histograms(arrays["seq_shapes"], arrays["label_seq_shapes"], max_grid_size=max_grid_size, chunk_size=chunk_size)

    print(f"[{split}] identifier histograms")
    identifiers_stats = identifier_histograms(arrays["puzzle_identifiers"], identifiers)

    print(f"[{split}] token histograms ({'exact' if EXACT_TOKEN_HIST else 'sampled'})")
    input_token_hist, input_token_info = token_histogram(
        arrays["inputs"],
        vocab_size=metadata.get("vocab_size", ARC_VOCAB_SIZE),
        exact=EXACT_TOKEN_HIST,
        sample_size=TOKEN_SAMPLE_SIZE,
        window_size=TOKEN_SAMPLE_WINDOW_SIZE,
        rng=rng,
    )
    label_token_hist, label_token_info = token_histogram(
        arrays["labels"],
        vocab_size=metadata.get("vocab_size", ARC_VOCAB_SIZE),
        exact=EXACT_TOKEN_HIST,
        sample_size=TOKEN_SAMPLE_SIZE,
        window_size=TOKEN_SAMPLE_WINDOW_SIZE,
        rng=rng,
    )

    return {
        "split": split,
        "metadata": metadata,
        "num_groups": int(len(arrays["group_indices"]) - 1),
        "num_puzzles": int(len(arrays["puzzle_identifiers"])),
        "num_examples": int(len(arrays["seq_shapes"])),
        "input_tokens_total": int(arrays["inputs"].shape[0]),
        "label_tokens_total": int(arrays["labels"].shape[0]),
        "puzzles_per_group_hist": count_hist_from_values(group_counts),
        "examples_per_group_hist": count_hist_from_values(group_example_counts),
        "examples_per_puzzle_hist": count_hist_from_values(puzzle_counts),
        "input_length_hist": input_length_hist,
        "label_length_hist": label_length_hist,
        "shapes": shapes,
        "identifiers": identifiers_stats,
        "input_token_hist": input_token_hist,
        "label_token_hist": label_token_hist,
        "input_token_info": input_token_info,
        "label_token_info": label_token_info,
    }

## Plot helpers

In [ ]:
def weighted_quantile_from_counts(counts, q, offset=0):
    counts = np.asarray(counts, dtype=np.int64)
    total = int(counts.sum())
    if total == 0:
        return np.nan
    target = q * (total - 1)
    idx = int(np.searchsorted(np.cumsum(counts), target + 1, side="left"))
    return idx - offset


def describe_counts(counts, offset=0):
    counts = np.asarray(counts, dtype=np.int64)
    total = int(counts.sum())
    if total == 0:
        return {"count": 0}
    values = np.arange(counts.size, dtype=np.float64) - float(offset)
    mean = float((values * counts).sum() / total)
    std = float(np.sqrt((((values - mean) ** 2) * counts).sum() / total))
    nonzero = np.flatnonzero(counts)
    return {
        "count": total,
        "mean": mean,
        "std": std,
        "min": int(nonzero[0] - offset),
        "p50": weighted_quantile_from_counts(counts, 0.50, offset),
        "p90": weighted_quantile_from_counts(counts, 0.90, offset),
        "p95": weighted_quantile_from_counts(counts, 0.95, offset),
        "p99": weighted_quantile_from_counts(counts, 0.99, offset),
        "max": int(nonzero[-1] - offset),
    }


def split_summary_table(stats_by_split):
    rows = []
    for split, stats in stats_by_split.items():
        examples_per_puzzle = describe_counts(stats["examples_per_puzzle_hist"])
        puzzles_per_group = describe_counts(stats["puzzles_per_group_hist"])
        examples_per_group = describe_counts(stats["examples_per_group_hist"])
        shape_stats = stats["shapes"]
        rows.append({
            "split": split,
            "groups": stats["num_groups"],
            "puzzles": stats["num_puzzles"],
            "examples": stats["num_examples"],
            "input_tokens_total": stats["input_tokens_total"],
            "label_tokens_total": stats["label_tokens_total"],
            "mean_puzzles_per_group": puzzles_per_group["mean"],
            "mean_examples_per_group": examples_per_group["mean"],
            "mean_examples_per_puzzle": examples_per_puzzle["mean"],
            "same_shape_ratio": shape_stats["same_shape_count"] / max(1, shape_stats["num_examples"]),
            "same_area_ratio": shape_stats["same_area_count"] / max(1, shape_stats["num_examples"]),
            "token_hist_mode": stats["input_token_info"]["mode"],
            "input_tokens_analyzed": stats["input_token_info"]["tokens_analyzed"],
            "label_tokens_analyzed": stats["label_token_info"]["tokens_analyzed"],
        })
    return pd.DataFrame(rows).set_index("split")


def distribution_table(stats):
    shape_stats = stats["shapes"]
    rows = {
        "puzzles_per_group": describe_counts(stats["puzzles_per_group_hist"]),
        "examples_per_group": describe_counts(stats["examples_per_group_hist"]),
        "examples_per_puzzle": describe_counts(stats["examples_per_puzzle_hist"]),
        "input_seq_len": describe_counts(stats["input_length_hist"]),
        "label_seq_len": describe_counts(stats["label_length_hist"]),
        "input_area": describe_counts(shape_stats["input_area"]),
        "output_area": describe_counts(shape_stats["output_area"]),
        "output_minus_input_area": describe_counts(shape_stats["area_delta"], offset=shape_stats["area_delta_offset"]),
    }
    return pd.DataFrame(rows).T


def plot_count_distribution(ax, counts, title, xlabel, offset=0, log_y=False):
    counts = np.asarray(counts, dtype=np.int64)
    nonzero = np.flatnonzero(counts)
    if nonzero.size == 0:
        ax.set_title(title)
        return
    lo, hi = int(nonzero[0]), int(nonzero[-1])
    x = np.arange(lo, hi + 1) - offset
    y = counts[lo : hi + 1]
    if len(x) <= 80:
        ax.bar(x, y, width=0.9, color="#4C78A8")
    else:
        ax.plot(x, y, color="#4C78A8", linewidth=1.5)
        ax.fill_between(x, y, color="#4C78A8", alpha=0.18)
    if log_y:
        ax.set_yscale("log", base=10)
        ax.grid(True, which="both", axis="y", alpha=0.25)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("count")


def plot_counter(ax, counter, title, top_n=TOP_N, horizontal=False):
    items = counter.most_common(top_n)
    if not items:
        ax.set_title(title)
        return
    labels, values = zip(*items)
    labels = [str(label) for label in labels]
    if horizontal:
        ax.barh(labels[::-1], values[::-1], color="#59A14F")
        ax.set_xlabel("puzzles")
    else:
        ax.bar(labels, values, color="#59A14F")
        ax.tick_params(axis="x", rotation=45)
        ax.set_ylabel("puzzles")
    ax.set_title(title)


def plot_token_hist(ax, counts, title):
    xs = np.arange(len(counts))
    labels = [ARC_TOKEN_NAMES.get(int(x), str(int(x))) for x in xs]
    ax.bar(xs, counts, color="#F28E2B")
    ax.set_xticks(xs)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_ylabel("tokens")
    ax.set_title(title)


def plot_heatmap(ax, matrix, title):
    shown = np.log1p(np.asarray(matrix, dtype=np.float64))
    im = ax.imshow(shown, origin="lower", cmap="viridis", interpolation="nearest")
    ax.set_title(title)
    ax.set_xlabel("width")
    ax.set_ylabel("height")
    ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="log1p(count)")


def show_split_report(stats):
    split = stats["split"]
    print(f"=== {split} ===")
    display(distribution_table(stats))

    fig, axes = plt.subplots(2, 3, figsize=(16, 8), constrained_layout=True)
    plot_count_distribution(axes[0, 0], stats["puzzles_per_group_hist"], "puzzles per group", "puzzles", log_y=False)
    plot_count_distribution(axes[0, 1], stats["examples_per_group_hist"], "examples per group", "examples", log_y=True)
    plot_count_distribution(axes[0, 2], stats["examples_per_puzzle_hist"], "examples per puzzle", "examples", log_y=True)
    plot_count_distribution(axes[1, 0], stats["input_length_hist"], "input sequence length", "tokens", log_y=True)
    plot_count_distribution(axes[1, 1], stats["label_length_hist"], "label sequence length", "tokens", log_y=True)
    plot_count_distribution(axes[1, 2], stats["shapes"]["area_delta"], "output area - input area", "cells", offset=stats["shapes"]["area_delta_offset"], log_y=True)
    fig.suptitle(f"{split}: group / puzzle / sequence distributions", fontsize=14)
    plt.show()

    fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), constrained_layout=True)
    plot_count_distribution(axes[0], stats["shapes"]["input_area"], "input area", "cells", log_y=True)
    plot_count_distribution(axes[1], stats["shapes"]["output_area"], "output area", "cells", log_y=True)
    plot_count_distribution(axes[2], stats["shapes"]["input_h"], "input height", "height", log_y=True)
    fig.suptitle(f"{split}: grid size distributions", fontsize=14)
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    plot_heatmap(axes[0], stats["shapes"]["input_dim_heatmap"], "input height x width")
    plot_heatmap(axes[1], stats["shapes"]["output_dim_heatmap"], "output height x width")
    fig.suptitle(f"{split}: shape heatmaps", fontsize=14)
    plt.show()

    fig, axes = plt.subplots(2, 2, figsize=(15, 9), constrained_layout=True)
    plot_token_hist(axes[0, 0], stats["input_token_hist"], f"input token distribution ({stats['input_token_info']['mode']})")
    plot_token_hist(axes[0, 1], stats["label_token_hist"], f"label token distribution ({stats['label_token_info']['mode']})")
    plot_counter(axes[1, 0], stats["identifiers"]["source_like"], "source-like category")
    plot_counter(axes[1, 1], stats["identifiers"]["transforms"], "augmentation transform")
    fig.suptitle(f"{split}: tokens and identifiers", fontsize=14)
    plt.show()

    fig, ax = plt.subplots(figsize=(10, max(5, TOP_N * 0.22)), constrained_layout=True)
    plot_counter(ax, stats["identifiers"]["base_tasks"], f"{split}: top base tasks by puzzle count", horizontal=True)
    plt.show()

## Compute statistics

`SPLITS` に入っている split を順番に集計します。train は shape/offset の全件集計を行うので、環境によっては数分かかります。

In [ ]:
identifiers = load_identifiers(DATASET_DIR)
stats_by_split = {}

for split in SPLITS:
    stats_by_split[split] = compute_split_stats(split, identifiers, chunk_size=CHUNK_SIZE)
    print(f"[{split}] done")

## Summary

In [ ]:
summary = split_summary_table(stats_by_split)
display(summary)

## Split reports

In [ ]:
for split, stats in stats_by_split.items():
    show_split_report(stats)

## Train origin subset reports

`build_arc_dataset.py` と同じ元データ prefix の challenge JSON を使い、train split に含まれる puzzle identifier を `training` / `evaluation` / `concept` の元subsetへ割り当てて集計します。`evaluation` は test task の train examples が train split に入っている部分です。

In [ ]:
train_origin_stats = {}
if ENABLE_TRAIN_ORIGIN_BREAKDOWN:
    train_origin_stats = compute_train_origin_breakdown(
        identifiers,
        input_file_prefix=ORIGIN_INPUT_FILE_PREFIX,
        subsets=TRAIN_ORIGIN_SUBSETS,
        chunk_size=CHUNK_SIZE,
    )
    train_origin_summary = split_summary_table(train_origin_stats)
    display(train_origin_summary)

    for origin, stats in train_origin_stats.items():
        show_split_report(stats)
else:
    print("Set ENABLE_TRAIN_ORIGIN_BREAKDOWN = True to compute train origin subset statistics.")

## Optional: compare train and test

両方の split を集計した場合に、主要な件数と平均値を横並びで確認します。

In [ ]:
if len(stats_by_split) >= 2:
    display(summary[[
        "groups",
        "puzzles",
        "examples",
        "mean_puzzles_per_group",
        "mean_examples_per_group",
        "mean_examples_per_puzzle",
        "same_shape_ratio",
        "same_area_ratio",
    ]])

    compare_metrics = ["groups", "puzzles", "examples", "mean_examples_per_puzzle", "same_shape_ratio", "same_area_ratio"]
    axes = summary[compare_metrics].T.plot(kind="bar", subplots=True, layout=(1, len(summary)), figsize=(5 * len(summary), 4), legend=False, rot=45)
    axes = np.asarray(axes).ravel()
    for ax, split in zip(axes, summary.index):
        ax.set_title(split)
        ax.set_ylabel("value")
    plt.tight_layout()
    plt.show()
else:
    print("Set SPLITS to include both 'train' and 'test' to show split comparison.")

## Optional: exact token histograms

正確な token 分布を取りたい場合は、最初の設定セルで `EXACT_TOKEN_HIST = True` にして再実行してください。train では `all__inputs.npy` と `all__labels.npy` の全体を読むため時間がかかります。